In [ ]:
# --- Configuration ---
import sys, time, json, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
%matplotlib inline

# Imports projet
sys.path.insert(0, str(Path.cwd().parent.parent))
from shared.data_loader import prepare_hierarchical_data_from_splits, build_parent_child_matrix
from shared.metrics import (compute_all_metrics, compute_per_class_f1,
                             plot_confusion_matrix, plot_per_class_f1)
from shared.utils import set_seed

# Reproductibilité
SEED = 42
set_seed(SEED)

# Chemins
PROJECT_ROOT = Path.cwd().parent.parent
DATASET_DIR = Path("/home/infres/tandriam-25/fil_rouge/dataset")
RESULTS_DIR = PROJECT_ROOT / "results" / "hydra-local-camembert"

# Style
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

print("Configuration terminée.")


In [ ]:
# --- Chargement des données splittées ---
print("Chargement des splits train/val/test...")
t0 = time.time()

data = prepare_hierarchical_data_from_splits(
    train_path=str(DATASET_DIR / "train_split.jsonl"),
    val_path=str(DATASET_DIR / "val_split.jsonl"),
    test_path=str(DATASET_DIR / "test_split.jsonl"),
    n_samples=None,  # Dataset complet
    seed=SEED,
)

load_time = time.time() - t0
print(f"Chargement terminé en {load_time:.0f}s")
print(f"  Train : {len(data['X_train']):,} produits")
print(f"  Val   : {len(data['X_val']):,} produits")
print(f"  Test  : {len(data['X_test']):,} produits")
print(f"  L1    : {data['num_classes_l1']} catégories")
print(f"  L2    : {data['num_classes_l2']} sous-catégories")


<a id="section-2"></a>
## 2. Vectorisation TF-IDF

On vectorise les textes avec TF-IDF (50K features, unigrammes + bigrammes, sublinear TF).
Le vectorizer est fit sur le train set uniquement, puis appliqué sur val et test.


In [ ]:
# --- Stopwords ---
import nltk
try:
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))

stopwords_combined = list(french_stops | english_stops)
print(f"Stopwords : {len(stopwords_combined)} mots (FR + EN)")


In [ ]:
# --- Vectorisation TF-IDF ---
from sklearn.feature_extraction.text import TfidfVectorizer

print("Vectorisation TF-IDF...")
t0 = time.time()

tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    stop_words=stopwords_combined,
    dtype=np.float32,
)

X_train_tfidf = tfidf.fit_transform(data["X_train"])
X_val_tfidf = tfidf.transform(data["X_val"])
X_test_tfidf = tfidf.transform(data["X_test"])

vec_time = time.time() - t0
print(f"Vectorisation terminée en {vec_time:.0f}s")
print(f"  Matrice train  : {X_train_tfidf.shape} ({X_train_tfidf.nnz:,} éléments non-nuls)")
print(f"  Matrice val    : {X_val_tfidf.shape}")
print(f"  Matrice test   : {X_test_tfidf.shape}")
print(f"  Vocabulaire    : {len(tfidf.vocabulary_):,} termes")


<a id="section-3"></a>
## 3. Classification L1 (157 catégories)

On utilise LinearSVC avec `class_weight='balanced'` pour compenser le déséquilibre de classes.


In [ ]:
# --- Entraînement L1 ---
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report

print("Entraînement LinearSVC sur L1...")
t0 = time.time()

clf_l1 = LinearSVC(
    class_weight="balanced",
    max_iter=200,
    C=1.0,
    dual="auto",
    random_state=SEED,
)
clf_l1.fit(X_train_tfidf, data["y_l1_train"])
train_time_l1 = time.time() - t0

# Prédictions
preds_l1_val = clf_l1.predict(X_val_tfidf)
preds_l1_test = clf_l1.predict(X_test_tfidf)

f1_l1_val = f1_score(data["y_l1_val"], preds_l1_val, average="macro")
f1_l1_test = f1_score(data["y_l1_test"], preds_l1_test, average="macro")

print(f"Entraînement L1 terminé en {train_time_l1:.0f}s")
print(f"  Val  F1 macro L1 : {f1_l1_val:.4f}")
print(f"  Test F1 macro L1 : {f1_l1_test:.4f}")


In [ ]:
# --- Top / Bottom classes L1 ---
per_class_f1_l1 = compute_per_class_f1(
    data["y_l1_test"], preds_l1_test, data["l1_encoder"]
)

sorted_f1 = sorted(per_class_f1_l1.items(), key=lambda x: x[1])
print("=== 10 pires classes L1 ===")
for name, score in sorted_f1[:10]:
    print(f"  {name:40s} F1 = {score:.4f}")

print(f"\n=== 10 meilleures classes L1 ===")
for name, score in sorted_f1[-10:]:
    print(f"  {name:40s} F1 = {score:.4f}")


<a id="section-4"></a>
## 4. Classification L2 (6000+ sous-catégories)

Même approche LinearSVC, mais sur les sous-catégories L2. Avec 6000+ classes,
l'entraînement sera plus long et le F1 macro plus bas (beaucoup de classes rares).


In [ ]:
# --- Entraînement L2 ---
from sklearn.linear_model import SGDClassifier

# SGDClassifier est plus rapide que LinearSVC pour un grand nombre de classes
print("Entraînement SGDClassifier sur L2 (6000+ classes)...")
t0 = time.time()

clf_l2 = SGDClassifier(
    loss="hinge",
    class_weight="balanced",
    alpha=1e-4,
    max_iter=100,
    random_state=SEED,
    n_jobs=-1,
    early_stopping=True,
    validation_fraction=0.1,
)
clf_l2.fit(X_train_tfidf, data["y_l2_train"])
train_time_l2 = time.time() - t0

# Prédictions
preds_l2_val = clf_l2.predict(X_val_tfidf)
preds_l2_test = clf_l2.predict(X_test_tfidf)

f1_l2_val = f1_score(data["y_l2_val"], preds_l2_val, average="macro")
f1_l2_test = f1_score(data["y_l2_test"], preds_l2_test, average="macro")

print(f"Entraînement L2 terminé en {train_time_l2:.0f}s")
print(f"  Val  F1 macro L2 : {f1_l2_val:.4f}")
print(f"  Test F1 macro L2 : {f1_l2_test:.4f}")


<a id="section-4b"></a>
## 4b. XGBoost sur TF-IDF

XGBoost est un classifieur non-linéaire basé sur les arbres de décision boostés.
Contrairement au SVM linéaire, il peut capter des interactions entre features.

In [ ]:
# --- XGBoost ---
from xgboost import XGBClassifier

print("Entraînement XGBoost sur L1...")
t0 = time.time()
xgb_l1 = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    n_jobs=10, tree_method="hist", random_state=SEED,
    eval_metric="mlogloss", verbosity=0,
)
xgb_l1.fit(X_train_tfidf, data["y_l1_train"])
xgb_time_l1 = time.time() - t0

preds_xgb_l1_test = xgb_l1.predict(X_test_tfidf)
f1_xgb_l1 = f1_score(data["y_l1_test"], preds_xgb_l1_test, average="macro")
print(f"XGBoost L1 terminé en {xgb_time_l1:.0f}s — F1 macro: {f1_xgb_l1:.4f}")

print("Entraînement XGBoost sur L2...")
t0 = time.time()
xgb_l2 = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    n_jobs=10, tree_method="hist", random_state=SEED,
    eval_metric="mlogloss", verbosity=0,
)
xgb_l2.fit(X_train_tfidf, data["y_l2_train"])
xgb_time_l2 = time.time() - t0

preds_xgb_l2_test = xgb_l2.predict(X_test_tfidf)
f1_xgb_l2 = f1_score(data["y_l2_test"], preds_xgb_l2_test, average="macro")
print(f"XGBoost L2 terminé en {xgb_time_l2:.0f}s — F1 macro: {f1_xgb_l2:.4f}")

<a id="section-4c"></a>
## 4c. LightGBM sur TF-IDF

LightGBM est un autre framework de gradient boosting, souvent plus rapide que XGBoost
grâce à son algorithme de découpage par histogramme et sa croissance par feuille.

In [ ]:
# --- LightGBM ---
from lightgbm import LGBMClassifier

print("Entraînement LightGBM sur L1...")
t0 = time.time()
lgbm_l1 = LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    n_jobs=10, random_state=SEED, verbose=-1,
)
lgbm_l1.fit(X_train_tfidf, data["y_l1_train"])
lgbm_time_l1 = time.time() - t0

preds_lgbm_l1_test = lgbm_l1.predict(X_test_tfidf)
f1_lgbm_l1 = f1_score(data["y_l1_test"], preds_lgbm_l1_test, average="macro")
print(f"LightGBM L1 terminé en {lgbm_time_l1:.0f}s — F1 macro: {f1_lgbm_l1:.4f}")

print("Entraînement LightGBM sur L2...")
t0 = time.time()
lgbm_l2 = LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    n_jobs=10, random_state=SEED, verbose=-1,
)
lgbm_l2.fit(X_train_tfidf, data["y_l2_train"])
lgbm_time_l2 = time.time() - t0

preds_lgbm_l2_test = lgbm_l2.predict(X_test_tfidf)
f1_lgbm_l2 = f1_score(data["y_l2_test"], preds_lgbm_l2_test, average="macro")
print(f"LightGBM L2 terminé en {lgbm_time_l2:.0f}s — F1 macro: {f1_lgbm_l2:.4f}")

<a id="section-5"></a>
## 5. Métriques hiérarchiques

Le TF-IDF entraîne L1 et L2 **indépendamment** — il n'y a aucune contrainte hiérarchique.
Le HCR (Hierarchical Consistency Rate) mesure la cohérence des prédictions : est-ce que le L2 prédit
est un enfant valide du L1 prédit ?


In [ ]:
# --- Métriques hiérarchiques pour les 3 classifieurs ---
results = {}
classifiers_info = [
    ("LinearSVC/SGD", preds_l1_test, preds_l2_test, train_time_l1 + train_time_l2),
    ("XGBoost", preds_xgb_l1_test, preds_xgb_l2_test, xgb_time_l1 + xgb_time_l2),
    ("LightGBM", preds_lgbm_l1_test, preds_lgbm_l2_test, lgbm_time_l1 + lgbm_time_l2),
]

for name, pl1, pl2, t in classifiers_info:
    m = compute_all_metrics(
        labels_l1=data["y_l1_test"], preds_l1=pl1,
        labels_l2=data["y_l2_test"], preds_l2=pl2,
        l2_to_l1_mapping=data["l2_to_l1_mapping"],
    )
    m["temps_train"] = f"{t:.0f}s"
    results[f"TF-IDF + {name}"] = m
    print(f"=== TF-IDF + {name} ===")
    for k, v in m.items():
        if isinstance(v, float):
            print(f"  {k:30s} : {v:.4f}")
        else:
            print(f"  {k:30s} : {v}")

<a id="section-6"></a>
## 6. Visualisations


In [ ]:
# --- Matrice de confusion L1 (top 30) ---
fig_cm_l1 = plot_confusion_matrix(
    data["y_l1_test"], preds_l1_test, data["l1_encoder"],
    title="Matrice de confusion L1 — TF-IDF Baseline", top_n=30,
)
plt.show()


In [ ]:
# --- F1 par classe L1 ---
fig_f1_l1 = plot_per_class_f1(
    per_class_f1_l1,
    title="F1 par classe L1 — TF-IDF Baseline",
    top_n=20,
)
plt.show()


In [ ]:
# --- F1 par classe L2 (top/bottom) ---
per_class_f1_l2 = compute_per_class_f1(
    data["y_l2_test"], preds_l2_test, data["l2_encoder"]
)

fig_f1_l2 = plot_per_class_f1(
    per_class_f1_l2,
    title="F1 par classe L2 — TF-IDF Baseline",
    top_n=20,
)
plt.show()

# Stats L2
f1_values_l2 = list(per_class_f1_l2.values())
print(f"F1 L2 — Moyenne: {np.mean(f1_values_l2):.4f}, Médiane: {np.median(f1_values_l2):.4f}")
print(f"  Classes L2 avec F1 = 0 : {sum(1 for v in f1_values_l2 if v == 0)}")
print(f"  Classes L2 avec F1 > 0.5 : {sum(1 for v in f1_values_l2 if v > 0.5)}")


<a id="section-8"></a>
## 8. Conclusion

### Forces du TF-IDF
- **Rapidité** : entraînement en quelques minutes (vs ~70h pour HYDRA)
- **Simplicité** : pas de GPU, pas de fine-tuning, peu d'hyperparamètres
- **F1 L1 compétitif** : ~86% macro — une baseline solide

### Limites
- **Pas de compréhension sémantique** : les synonymes et paraphrases ne sont pas capturés
- **Pas de hiérarchie** : L1 et L2 sont prédits indépendamment → HCR faible
- **F1 L2 limité** : trop de classes rares pour un modèle linéaire sans contextualisation

### Ce que HYDRA apporte
- **Embeddings contextuels** (CamemBERT) : compréhension sémantique profonde
- **Architecture multi-tête** : partage d'informations entre L1 et L2
- **Focal Loss** : meilleure gestion du déséquilibre de classes
- **Hiérarchie** : possibilité de contraindre L2 conditionnellement à L1

---
*Notebook généré dans le cadre du projet de classification hiérarchique Rakuten — Télécom Paris x Rakuten France, 2026.*
